# Eco-Scale: Prediction of Safe Server Shutdowns\n## Time-Series & Cluster-Level Regression\n\nInstead of predicting binary load on a single server, this notebook predicts the **exact number of servers** out of a 1000-node cluster that can be safely powered down while keeping overall capacity at a safe 75% limit.

In [1]:
import pandas as pd\nimport numpy as np\nimport plotly.express as px\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, r2_score\nimport joblib

### 1. Data Loading

In [2]:
df = pd.read_csv('cluster_telemetry.csv')\ndf['timestamp'] = pd.to_datetime(df['timestamp'])\ndf.head()

### 2. Exploratory Data Analysis (EDA) & Feature Relationships

In [3]:
# Time-Series View of Cluster Load\nfig = px.line(df[:168], x='timestamp', y=['avg_cpu_load', 'safe_shutdown_count'], \n              title='1-Week Snapshot: Inverse Relationship between CPU Load & Servers to Shutdown')\nfig.show()\n\n# Correlation Heatmap\ncorr = df.drop('timestamp', axis=1).corr()\nfig2 = px.imshow(corr, text_auto=True, aspect=\"auto\", title=\"Feature Correlation Heatmap\")\nfig2.show()

### 3. Model Training (Random Forest Regressor)

In [4]:
features = ['hour_of_day', 'day_of_week', 'avg_cpu_load', 'avg_memory_load', 'network_traffic_gbps']\nX = df[features]\ny = df['safe_shutdown_count']\n\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)\n\nmodel = RandomForestRegressor(n_estimators=100, random_state=42)\nmodel.fit(X_train, y_train)

### 4. Evaluation & Feature Importance

In [5]:
preds = model.predict(X_test)\nprint(f\"Mean Absolute Error (MAE): {mean_absolute_error(y_test, preds):.2f} servers\")\nprint(f\"R^2 Score: {r2_score(y_test, preds):.4f}\")\n\nimportances = pd.DataFrame({\n    'Feature': features,\n    'Importance': model.feature_importances_\n}).sort_values(by='Importance', ascending=False)\n\nfig3 = px.bar(importances, x='Importance', y='Feature', orientation='h', title='Feature Importance in Random Forest Model')\nfig3.show()

### 5. Model Saving

In [6]:
joblib.dump(model, 'rf_cluster_model_notebook.pkl')\nprint(\"Model saved successfully.\")